# Polish Cities with NUTS 2 Regions

Builds a Spark DataFrame of Polish cities with their voivodeships and NUTS 2 region codes, enriched with **population**, **areas in km²** (city / gmina / powiat) and Polish administrative divisions — **Powiat** and **Gmina**.

Sources: `datasets/world-cities` (GitHub), the GeoNames PL dump (download.geonames.org — population, coordinates, TERYT codes) and official Geoportal 2025 boundaries (powiat/gmina polygons, via GitHub). The Wikidata SPARQL endpoint is unreachable (DNS-blocked) from this compute.

In [0]:
# ---- Polish cities: NUTS 2 regions + population, areas, Powiat & Gmina ----
# Sources (only GitHub raw and download.geonames.org are reachable from this
# compute; query.wikidata.org and most other endpoints fail DNS resolution):
#   1. datasets/world-cities (GitHub raw)  -> Polish city list + GeoNames IDs
#   2. GeoNames PL dump (download.geonames.org) -> population, coordinates and
#      TERYT-style admin codes (admin2 = powiat, admin3 = gmina)
#   3. Geoportal 2025 boundaries (GitHub: waszkiewiczja/GeoJSON-Polska-…)
#      -> official Polish powiat/gmina names + boundary polygons (areas)
# NUTS 2 codes follow the official NUTS 2021 classification (PL21..PL72).
# NOTE: Warszawa itself is missing from the world-cities dataset (only its
# districts appear), so it is added manually from GeoNames (id 756135).

import csv
import io
import math
import zipfile
from collections import defaultdict

import requests

HEADERS = {"User-Agent": "DatabricksNotebook/1.0 (data engineering query)"}

# ---------------------------------------------------------------------------
# Geodesic polygon area (spherical-excess algorithm, as in turf.js) -> km^2
# ---------------------------------------------------------------------------
R_EARTH = 6378137.0  # WGS84 equatorial radius, metres

def _ring_area_m2(ring):
    if len(ring) < 3:
        return 0.0
    total = 0.0
    for i in range(len(ring) - 1):
        lon1, lat1 = ring[i]
        lon2, lat2 = ring[i + 1]
        total += (math.radians(lon2) - math.radians(lon1)) * (
            2 + math.sin(math.radians(lat1)) + math.sin(math.radians(lat2))
        )
    return abs(total) * R_EARTH * R_EARTH / 2.0

def feature_area_km2(geom):
    polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    area = 0.0
    for poly in polygons:
        if not poly:
            continue
        a = _ring_area_m2(poly[0])          # outer ring
        for hole in poly[1:]:              # subtract interior holes
            a -= _ring_area_m2(hole)
        area += a
    return area / 1_000_000.0

# ---------------------------------------------------------------------------
# NUTS 2021: dataset voivodeship -> (NUTS 2 code, official Polish name)
# ---------------------------------------------------------------------------
NUTS_MAP = {
    "Greater Poland":       ("PL41", "Wielkopolskie"),
    "Kujawsko-Pomorskie":   ("PL61", "Kujawsko-pomorskie"),
    "Lesser Poland":        ("PL21", "Małopolskie"),
    "Lodz Voivodeship":     ("PL71", "Łódzkie"),
    "Lower Silesia":        ("PL51", "Dolnośląskie"),
    "Lublin":               ("PL31", "Lubelskie"),
    "Lubusz":               ("PL43", "Lubuskie"),
    "Mazovia":              ("PL34", "Mazowieckie"),
    "Opole Voivodeship":    ("PL52", "Opolskie"),
    "Podlasie":             ("PL32", "Podlaskie"),
    "Pomerania":            ("PL63", "Pomorskie"),
    "Silesia":              ("PL72", "Śląskie"),
    "Subcarpathia":         ("PL22", "Podkarpackie"),
    "Swietokrzyskie":       ("PL33", "Świętokrzyskie"),
    "Warmia-Masuria":       ("PL62", "Warmińsko-mazurskie"),
    "West Pomerania":       ("PL42", "Zachodniopomorskie"),
}

# ---------------------------------------------------------------------------
# 1) Polish cities (world-cities) + Warszawa added manually
# ---------------------------------------------------------------------------
r = requests.get(
    "https://raw.githubusercontent.com/datasets/world-cities/master/data/world-cities.csv",
    headers=HEADERS, timeout=60,
)
r.raise_for_status()
cities = [row for row in csv.DictReader(io.StringIO(r.text)) if row["country"] == "Poland"]
cities.append({"name": "Warszawa", "country": "Poland", "subcountry": "Mazovia", "geonameid": "756135"})
print(f"City list: {len(cities)} entries (363 from world-cities + Warszawa)")

# ---------------------------------------------------------------------------
# 2) GeoNames PL dump: population, coordinates, TERYT admin codes
# ---------------------------------------------------------------------------
r = requests.get("https://download.geonames.org/export/dump/PL.zip", headers=HEADERS, timeout=120)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
GN_COLS = ["geonameid", "name", "asciiname", "alternatenames", "lat", "lng",
           "feat_class", "feat_code", "country", "cc2", "admin1", "admin2",
           "admin3", "admin4", "population", "elevation", "dem", "timezone", "moddate"]
geoid_set = {c["geonameid"] for c in cities}
geonames = {}
for line in z.read("PL.txt").decode("utf-8").splitlines():
    parts = line.split("\t")
    if parts[0] in geoid_set:
        geonames[parts[0]] = dict(zip(GN_COLS, parts))
print(f"GeoNames matched: {len(geonames)}/{len(cities)}")

# ---------------------------------------------------------------------------
# 3) Official powiat/gmina boundaries (Geoportal 2025, via GitHub)
# ---------------------------------------------------------------------------
GEOJSON_BASE = "https://raw.githubusercontent.com/waszkiewiczja/GeoJSON-Polska-Wojewodztwa-Powiaty-Gminy/main"

r = requests.get(f"{GEOJSON_BASE}/powiaty.json", headers=HEADERS, timeout=120)
r.raise_for_status()
powiaty = {
    f["properties"]["JPT_KOD_JE"]: (f["properties"]["JPT_NAZWA_"], feature_area_km2(f["geometry"]))
    for f in r.json()["features"]
}

r = requests.get(f"{GEOJSON_BASE}/gminy.json", headers=HEADERS, timeout=120)
r.raise_for_status()
gmina_by_prefix = defaultdict(list)       # 6-digit TERYT prefix -> candidate gminas
gmina_by_name_powiat = defaultdict(list)  # (powiat code, lower name) -> candidate gminas
for f in r.json()["features"]:
    code = f["properties"]["JPT_KOD_JE"]  # 7 chars: 6-digit code + unit-type char
    gm = (code, f["properties"]["JPT_NAZWA_"], feature_area_km2(f["geometry"]), code[6])
    gmina_by_prefix[code[:6]].append(gm)
    gmina_by_name_powiat[(code[:4], f["properties"]["JPT_NAZWA_"].strip().lower())].append(gm)

GMINA_TYPE = {"1": "miejska", "2": "wiejska", "3": "miejsko-wiejska"}

def resolve_gmina(admin3, powiat_code, city_name):
    """Resolve a city's gmina.
    1) urban unit ('miejska') carrying the GeoNames admin3 code;
    2) fallback: same-named gmina miejska in the same powiat (fixes the ~10
       GeoNames admin3 codes that point at the gmina wiejska instead);
    3) otherwise the gmina miejsko-wiejska / any unit on that code.
    """
    cands = gmina_by_prefix.get(admin3 or "", [])
    for c in cands:
        if c[3] == "1":
            return c
    for c in gmina_by_name_powiat.get((powiat_code, city_name.strip().lower()), []):
        if c[3] == "1":
            return c
    if cands:
        for c in cands:
            if c[3] == "3":
                return c
        return cands[0]
    return None

# ---------------------------------------------------------------------------
# 4) Build the enriched dataset
# ---------------------------------------------------------------------------
rows = []
no_powiat, no_gmina = [], []
for c in cities:
    g = geonames.get(c["geonameid"], {})
    nuts = NUTS_MAP.get(c["subcountry"])
    powiat = powiaty.get(g.get("admin2", ""))
    gm = resolve_gmina(g.get("admin3", ""), g.get("admin2", ""), c["name"])
    if not powiat:
        no_powiat.append(c["name"])
    if not gm:
        no_gmina.append(c["name"])
    is_district = g.get("feat_code") == "PPLX"  # district of a bigger city, not a standalone city
    rows.append({
        "City":                     c["name"],
        "NUTS_Code":               nuts[0] if nuts else None,
        "NUTS_Region":             nuts[1] if nuts else None,
        "GeoRegion":               c["subcountry"],
        "AdminRegion_Voivodeship": nuts[1] if nuts else None,
        "Powiat":                  powiat[0] if powiat else None,   # np. "powiat bocheński"
        "Gmina":                   gm[1] if gm else None,          # np. "Bochnia"
        "Gmina_Type":              GMINA_TYPE.get(gm[3]) if gm else None,
        "Population":              int(g["population"]) if g.get("population") else None,
        # City area: exact where the city IS a gmina miejska (gmina == city);
        # NULL for districts and for towns inside gminy miejsko-wiejskie
        # (their city-proper polygon is not part of this boundary dataset).
        "Area_km2":                gm[2] if (gm and gm[3] == "1" and not is_district) else None,
        "Gmina_Area_km2":          gm[2] if gm else None,
        "Powiat_Area_km2":         powiat[1] if powiat else None,
        "Latitude":                float(g["lat"]) if g.get("lat") else None,
        "Longitude":               float(g["lng"]) if g.get("lng") else None,
        "GeoNames_ID":             c["geonameid"],
    })

schema = (
    "City STRING, NUTS_Code STRING, NUTS_Region STRING, GeoRegion STRING, "
    "AdminRegion_Voivodeship STRING, Powiat STRING, Gmina STRING, Gmina_Type STRING, "
    "Population INT, Area_km2 DOUBLE, Gmina_Area_km2 DOUBLE, Powiat_Area_km2 DOUBLE, "
    "Latitude DOUBLE, Longitude DOUBLE, GeoNames_ID STRING"
)
spark_df = spark.createDataFrame(rows, schema=schema)

# Register as a temporary view so you can query it with SQL
spark_df.createOrReplaceTempView("polish_cities")

n_area = sum(1 for x in rows if x["Area_km2"] is not None)
print(f"\nEnriched {len(rows)} cities | powiat missing: {len(no_powiat)} {no_powiat[:5]} | gmina missing: {len(no_gmina)} {no_gmina[:5]}")
print(f"Exact city area (gmina miejska): {n_area}/{len(rows)}; for the remaining cities Gmina_Area_km2 / Powiat_Area_km2 give the administrative context")

display(spark_df.orderBy(spark_df["Population"].desc(), "City"))